# Autonomous SDR Agent

This notebook implements a production-style Autonomous Sales Development Representative (SDR). It generates personalized subject lines and emails, manages conversation memory, correctly classifies replies, and decides on next actions using a State Machine logic.

## 1. Environment Setup

In [ ]:
import os
import json
import asyncio
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content

# Load environment variables
load_dotenv()

# Initialize clients
client = OpenAI()
sg_client = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
SENDER_EMAIL = os.environ.get('SENDER_EMAIL')

# --- Helper Decorator for Tools ---
# This ensures the notebook runs even if you don't have the 'agents' library installed
def function_tool(func):
    return func

## 2. SendGrid Configuration & Test

In [ ]:
def test_sendgrid():
    """Sends a test email to confirm SendGrid setup."""
    if not SENDER_EMAIL:
        print('SENDER_EMAIL not set. Skipping test.')
        return
    try:
        from_email = Email(SENDER_EMAIL)
        to_email = To(SENDER_EMAIL)  # Send to self
        subject = 'Test Email from Autonomous SDR Agent'
        content = Content('text/plain', 'SendGrid is configured correctly.')
        mail = Mail(from_email, to_email, subject, content)
        response = sg_client.client.mail.send.post(request_body=mail.get())
        if response.status_code == 202:
            print('✅ SendGrid test successful. Check your inbox.')
        else:
            print(f'❌ SendGrid test failed with status code: {response.status_code}')
    except Exception as e:
        print(f'❌ SendGrid test failed: {e}')

test_sendgrid()

## 3. Memory Model

In [ ]:
conversation_memory = {}

def initialize_memory(prospect_id: str, prospect_info: dict):
    if prospect_id not in conversation_memory:
        conversation_memory[prospect_id] = {
            'prospect_info': prospect_info,
            'emails_sent': [],
            'replies_received': [],
            'classification': 'Not Contacted',
            'status': 'Pending',
            'last_action': None
        }

## 4. Tool Definitions

In [ ]:
@function_tool
def send_email(prospect_id: str, subject: str, body: str):
    """Sends an email to a prospect using SendGrid."""
    try:
        prospect_email = conversation_memory[prospect_id]['prospect_info']['email']
        from_email = Email(SENDER_EMAIL)
        to_email = To(prospect_email)
        content = Content('text/plain', body)
        mail = Mail(from_email, to_email, subject, content)
        sg_client.client.mail.send.post(request_body=mail.get())
        log_decision(prospect_id, 'Email Sent', f'Subject: {subject}')
        return {'status': 'success'}
    except Exception as e:
        return {'status': 'error', 'reason': str(e)}

@function_tool
def store_memory(prospect_id: str, data: dict):
    """Updates the conversation memory."""
    if prospect_id in conversation_memory:
        conversation_memory[prospect_id].update(data)
        return {'status': 'success'}
    return {'status': 'error', 'reason': 'Prospect not found'}

@function_tool
def load_memory(prospect_id: str):
    """Loads the conversation memory."""
    return conversation_memory.get(prospect_id, None)

@function_tool
def log_decision(prospect_id: str, step: str, decision: str):
    """Logs a decision made by the SDR Manager."""
    log_entry = f"{datetime.now().strftime('%H:%M:%S')} | ID: {prospect_id} | Step: {step} | Decision: {decision}"
    print(f'[SDR_TRACE] {log_entry}')
    return {'status': 'logged'}

## 5. Agent Definitions (With System Prompts)

**FIX APPLIED:** Moved instructions to the `system` role for better reliability.

In [ ]:
async def outbound_writer_agent(prospect_info: dict):
    # Generate both subject and body
    system_prompt = "You are an expert SDR. Write a compelling cold email. Output ONLY JSON in this format: {\"subject\": \"...\", \"body\": \"...\"}"
    user_prompt = f"Write an email to: {json.dumps(prospect_info)}"
    
    response = await client.chat.completions.create(
        model='gpt-4o-mini', 
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ],
        response_format={ "type": "json_object" } # Force JSON output
    )
    return json.loads(response.choices[0].message.content)

async def reply_classifier_agent(reply: str):
    system_prompt = "You are a sales classification assistant. Classify the reply into exactly one of these categories: 'Interested', 'Objection', 'Not Interested', 'Out of Office'. Output only the category name."
    user_prompt = f"Reply: {reply}"
    
    response = await client.chat.completions.create(
        model='gpt-4o-mini', 
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ]
    )
    return response.choices[0].message.content.strip()

async def follow_up_writer_agent(conversation_history: dict):
    system_prompt = "You are an expert SDR. Write a polite, convincing follow-up email to address the prospect's last objection."
    user_prompt = f"History: {json.dumps(conversation_history)}"
    
    response = await client.chat.completions.create(
        model='gpt-4o-mini', 
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ]
    )
    return response.choices[0].message.content

async def escalation_agent(conversation_history: dict):
    system_prompt = "You are a Sales Manager. Summarize this conversation for a human Account Executive. Include key interest signals and next steps."
    user_prompt = f"History: {json.dumps(conversation_history)}"
    
    response = await client.chat.completions.create(
        model='gpt-4o-mini', 
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ]
    )
    return response.choices[0].message.content

## 6. SDR Manager & Workflow Orchestration

**FIX APPLIED:** Updated classification logic to be strict (preventing false positives) and used dynamic subject lines.

In [ ]:
async def run_sdr_workflow(prospect_id: str, prospect_info: dict, reply_simulation: str = None):
    # 1. Initialize Memory
    initialize_memory(prospect_id, prospect_info)
    log_decision(prospect_id, 'Start', 'Workflow initiated.')

    # 2. Outbound Email Flow
    log_decision(prospect_id, 'Outbound', 'Generating initial email.')
    email_content = await outbound_writer_agent(prospect_info)
    
    # Now we use the AI-generated subject
    send_email(prospect_id, email_content['subject'], email_content['body'])
    store_memory(prospect_id, {'status': 'Contacted', 'emails_sent': [email_content], 'last_action': 'Initial outreach sent.'})

    # 3. Simulate and Classify Reply
    if not reply_simulation:
        log_decision(prospect_id, 'End', 'No reply simulated. Workflow complete.')
        return
    
    log_decision(prospect_id, 'Reply Received', reply_simulation)
    classification = await reply_classifier_agent(reply_simulation)
    store_memory(prospect_id, {'replies_received': [reply_simulation], 'classification': classification, 'status': 'Replied'})
    log_decision(prospect_id, 'Classification', f'Reply classified as: {classification}')

    # 4. Decision Engine (Corrected Logic)
    if classification == 'Interested':
        log_decision(prospect_id, 'Decision', 'Prospect is interested. Escalating to human.')
        handoff_summary = await escalation_agent(load_memory(prospect_id))
        store_memory(prospect_id, {'status': 'Escalated', 'last_action': handoff_summary})
        print(f'\n🚀 ESCALATION SUMMARY:\n{handoff_summary}\n')
        
    elif classification == 'Objection':
        log_decision(prospect_id, 'Decision', 'Prospect has an objection. Generating follow-up.')
        follow_up_body = await follow_up_writer_agent(load_memory(prospect_id))
        send_email(prospect_id, 'Re: ' + email_content['subject'], follow_up_body)
        store_memory(prospect_id, {'status': 'Follow-up Sent', 'last_action': 'Objection follow-up sent.'})
        
    else: # Not Interested, Out of Office
        log_decision(prospect_id, 'Decision', 'Prospect not interested/OOO. Stopping workflow.')
        store_memory(prospect_id, {'status': 'Closed', 'last_action': 'Workflow stopped.'})

    log_decision(prospect_id, 'End', 'Workflow complete.')

## 7. End-to-End Demo Run

In [ ]:
prospect = {
    'id': 'prospect_001',
    'info': {
        'name': 'Samandari',
        'title': 'CTO',
        'company': 'Innovate Inc.',
        'email': 'cezaremardini10@gmail.com.com'
    }
}

# Scenario 1: Interested
print("\n--- SCENARIO 1: Interested Reply ---")
await run_sdr_workflow(prospect['id'], prospect['info'], "This sounds interesting. Can you send pricing?")

# Scenario 2: Objection (Resetting ID for demo)
print("\n--- SCENARIO 2: Objection Reply ---")
prospect['id'] = 'prospect_002'
await run_sdr_workflow(prospect['id'], prospect['info'], "We don't have budget for this right now.")